# Analyse Classification performance

0. Annotate data sample of 870 rels by file (~50 per goal)

1. **Metric vs. Manual annotation** (overall and by goal):
    - Recall
    - Precision
    - Accuracy
    - F1-score
    - Coverage Ratio: (# of distinct reference arguments covered) ÷ (total reference arguments).
    - Overgeneration: (# retrieved – # matched) ÷ (# retrieved).





In [2]:
import os, re, random, sys, glob
import pandas as pd

## Relationships sampling and annotations

1) For each goal g, choose up to 'target_per_goal' distinct arg1 (id_arg1) of that goal.
2) For each other goal h != g, pick one row that matches (g->h) and that arg1 (if any), else any row (g->h).
3) Top up randomly (no duplicates) to reach 'target_total' if possible.

In [5]:
LABEL_MAP = {
    "s": "Support",
    "a": "Attack",
    "r": "Rephrase",
    "n": "No Relationship",
}
VALID_KEYS = set(LABEL_MAP.keys())

def parse_goal(id_str):
    try:
        return int(str(id_str).split("_", 1)[0])
    except Exception:
        return None

def _select_pool_cross_goal(df, seed=42, target_per_goal=5, target_total=1500):
    rng = random.Random(seed)

    # add helper columns once
    if "g1" not in df.columns or "g2" not in df.columns:
        df = df.copy()
        df["g1"] = df["id_arg1"].apply(parse_goal)
        df["g2"] = df["id_arg2"].apply(parse_goal)

    # universe of goals seen in the file (usually 0..17)
    goals = sorted(set(df["g1"].dropna().astype(int).unique().tolist()))
    picked_idx = []

    # fast lookup by (g1,g2)
    by_pair = {(g1, g2): sub.index.to_list()
               for (g1, g2), sub in df.groupby(["g1", "g2"], dropna=False)}

    for g in goals:
        # distinct arg1 inside goal g
        arg1s_g = df.loc[df["g1"] == g, "id_arg1"].dropna().astype(str).unique().tolist()
        rng.shuffle(arg1s_g)
        chosen_arg1s = arg1s_g[:target_per_goal] if len(arg1s_g) >= target_per_goal else arg1s_g

        for a1 in chosen_arg1s:
            for h in goals:
                if h == g:
                    continue
                # prefer rows with this specific arg1 vs goal h
                mask = (df["g1"] == g) & (df["g2"] == h) & (df["id_arg1"].astype(str) == a1)
                rows = df[mask].index.to_list()
                if not rows:
                    # fallback: any row for (g->h)
                    rows = by_pair.get((g, h), [])
                if rows:
                    picked_idx.append(rng.choice(rows))

    # de-duplicate while preserving order
    seen = set()
    ordered = []
    for i in picked_idx:
        if i not in seen:
            seen.add(i)
            ordered.append(i)

    # top up to target_total if needed
    if len(ordered) < target_total:
        remaining = [i for i in df.index if i not in seen]
        rng.shuffle(remaining)
        ordered.extend(remaining[:max(0, target_total - len(ordered))])

    return df.loc[ordered].copy()

def parse_goal(id_str):
    # id format: "<goal>_<index>"
    try:
        return int(str(id_str).split("_", 1)[0])
    except Exception:
        return None

def sample_and_annotate_cross_goal(
    input_dir,
    output_dir,
    filename,
    seed=42,
    target_per_goal=5,
    target_total=1500,
    autosave_every=50,
):
    os.makedirs(output_dir, exist_ok=True)

    path = os.path.join(input_dir, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(path)

    # read raw file (allow empty strings, not NaN)
    df = pd.read_csv(path, keep_default_na=False)
    required = {"SDGarg1", "SDGarg2", "id_arg1", "id_arg2", "rel"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    base, ext = os.path.splitext(filename)
    out_sample = os.path.join(output_dir, f"{base}_sample{ext}")

    # resume or create sample
    if os.path.exists(out_sample):
        print(f"Resuming from sampled file: {out_sample}")
        sampled = pd.read_csv(out_sample, keep_default_na=False)
    else:
        sampled = _select_pool_cross_goal(df, seed=seed,
                                          target_per_goal=target_per_goal,
                                          target_total=target_total)
        if "rel" not in sampled.columns:
            sampled["rel"] = ""
        else:
            sampled["rel"] = sampled["rel"].fillna("")  # ensure blanks are blanks
        sampled.to_csv(out_sample, index=False, encoding="utf-8")
        print(f"Sample saved: {out_sample} (rows={len(sampled)})")

    print("\nAnnotation controls:")
    print("  Type S/A/R/N → Support/Attack/Rephrase/No Relationship")
    print("  Enter to skip, 'b' to go back, 'q' to save & quit.\n")

    # find first unlabeled row (treat NaN and empty as unlabeled)
    sampled["rel"] = sampled["rel"].astype(str)
    mask_unlabeled = sampled["rel"].isna() | (sampled["rel"].str.strip().isin({"", "nan", "None"}))
    unlabeled_idx = sampled.index[mask_unlabeled].tolist()
    if unlabeled_idx:
        i = sampled.index.get_loc(unlabeled_idx[0])
    else:
        print("All sampled rows already labeled.")
        return out_sample

    labeled_since_save = 0
    while 0 <= i < len(sampled):
        row = sampled.iloc[i]
        a1 = str(row["SDGarg1"])
        a2 = str(row["SDGarg2"])
        print(f"\n[{i+1}/{len(sampled)}]  G1={parse_goal(row['id_arg1'])}  G2={parse_goal(row['id_arg2'])}")
        print("Arg1:", a1)
        print("Arg2:", a2)
        prev = str(row.get("rel", "")).strip()
        if prev and prev.lower() not in {"nan", "none"}:
            print(f"(already labeled: {prev})")

        key = input("Label [S/A/R/N, Enter=skip, b=back, q=quit] > ").strip().lower()

        if key == "q":
            sampled.to_csv(out_sample, index=False, encoding="utf-8")
            print(f"\nSaved & quit: {out_sample}")
            return out_sample
        if key == "b":
            i = max(0, i - 1)
            continue
        if key == "":
            i += 1
            continue

        if key in VALID_KEYS:
            sampled.iat[i, sampled.columns.get_loc("rel")] = LABEL_MAP[key]
            labeled_since_save += 1
        else:
            print("Invalid key. Use S/A/R/N, Enter, b, or q.")
            continue

        if labeled_since_save >= autosave_every:
            sampled.to_csv(out_sample, index=False, encoding="utf-8")
            print(f"(autosaved {autosave_every} labels) -> {out_sample}")
            labeled_since_save = 0

        i += 1

    sampled.to_csv(out_sample, index=False, encoding="utf-8")
    print(f"\nDone. Saved: {out_sample}")
    return out_sample

In [6]:
input_dir = r"..\\Data\\Relationships Keywords"
prefix = 'GLOBAL_SGD2023_'

file_names = []
for file in glob.glob(os.path.join(input_dir, "*cross_goal*.csv")):
    if prefix in file.split("\\")[-1]:
        file_names.append(file.split("\\")[-1])
print(file_names)

['cross_goalGLOBAL_SGD2023_deepseek-r1-70b.csv', 'cross_goalGLOBAL_SGD2023_gemma3-27b.csv', 'cross_goalGLOBAL_SGD2023_gemma3-4b.csv', 'cross_goalGLOBAL_SGD2023_llama3.3-70b.csv', 'cross_goalGLOBAL_SGD2023_qwen2.5-3b.csv']


In [ ]:
input_dir = r"..\\Data\\Relationships Keywords"
output_dir = r"..\\Data\\Relationships Keywords\\Annotated"
prefix = 'GLOBAL_SGD2023_'

file_names = ['cross_goalGLOBAL_SGD2023_deepseek-r1-70b.csv', 
              'cross_goalGLOBAL_SGD2023_gemma3-27b.csv']

for document in file_names:
    sample_and_annotate_cross_goal(
        input_dir,
        output_dir,
        document,
        seed=42,
        target_per_goal=3,
        target_total=870,
        autosave_every=50)

Sample saved: ..\\Data\\Relationships Keywords\\Annotated\cross_goalGLOBAL_SGD2023_deepseek-r1-70b_sample.csv (rows=870)

Annotation controls:
  Type S/A/R/N → Support/Attack/Rephrase/No Relationship
  Enter to skip, 'b' to go back, 'q' to save & quit.


[1/870]  G1=0  G2=1
Arg1: We conclude by underscoring the vital, life-affirming importance of four key international agreements: the Sustainable Development Goals, the Paris Climate Agreement, the Kunming-Montreal Framework for Biodiversity, and the High Seas Treaty.
Arg2: Space-based technologies help address data gaps and timeliness, including supporting the 'leave no one behind' principle;

[2/870]  G1=0  G2=2
Arg1: We conclude by underscoring the vital, life-affirming importance of four key international agreements: the Sustainable Development Goals, the Paris Climate Agreement, the Kunming-Montreal Framework for Biodiversity, and the High Seas Treaty.
Arg2: 4. Sustainable ecosystems, sustainable agriculture, and climate resilience

### Intra-goal 

In [3]:
def _select_pool_intra_goal(
    df,
    seed=42,
    target_per_goal=25,          
    per_arg1_limit=1,            
    target_total=1500,
):
    rng = random.Random(seed)

    # add helper columns once
    if "g1" not in df.columns or "g2" not in df.columns:
        df = df.copy()
        df["g1"] = df["id_arg1"].apply(parse_goal)
        df["g2"] = df["id_arg2"].apply(parse_goal)

    # only rows where both args are from the same goal
    intra = df[(df["g1"].notna()) & (df["g2"].notna()) & (df["g1"] == df["g2"])].copy()
    intra["g1"] = intra["g1"].astype(int)
    goals = sorted(intra["g1"].unique().tolist())

    picked_idx = []

    # fast lookup: for each goal, group rows by arg1 within that goal
    by_goal_arg1 = {}
    for (g, a1), sub in intra.groupby(["g1", "id_arg1"], dropna=False):
        by_goal_arg1[(g, str(a1))] = sub.index.to_list()

    by_goal_any = {g: sub.index.to_list() for g, sub in intra.groupby("g1", dropna=False)}

    for g in goals:
        # distinct arg1 inside goal g
        arg1s_g = (
            intra.loc[intra["g1"] == g, "id_arg1"]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )
        rng.shuffle(arg1s_g)
        chosen_arg1s = arg1s_g[:target_per_goal] if len(arg1s_g) >= target_per_goal else arg1s_g

        for a1 in chosen_arg1s:
            # all intra-goal rows where this is the left argument
            rows = by_goal_arg1.get((g, a1), [])
            if rows:
                rng.shuffle(rows)
                take = rows[:per_arg1_limit] if per_arg1_limit else rows
                picked_idx.extend(take)
            else:
                pool = by_goal_any.get(g, [])
                if pool:
                    picked_idx.append(rng.choice(pool))

    # de-duplicate while preserving order
    seen = set()
    ordered = []
    for i in picked_idx:
        if i not in seen:
            seen.add(i)
            ordered.append(i)

    if len(ordered) < target_total:
        remaining = [i for i in intra.index if i not in seen]
        rng.shuffle(remaining)
        ordered.extend(remaining[:max(0, target_total - len(ordered))])

    return df.loc[ordered].copy()

def sample_and_annotate_intra_goal(
    input_dir,
    output_dir,
    filename,
    seed=42,
    target_per_goal=25,     
    per_arg1_limit=1,        
    target_total=1500,
    autosave_every=50):

    os.makedirs(output_dir, exist_ok=True)

    path = os.path.join(input_dir, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(path)

    # read raw file (allow empty strings, not NaN)
    df = pd.read_csv(path, keep_default_na=False)
    required = {"SDGarg1", "SDGarg2", "id_arg1", "id_arg2", "rel"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    base, ext = os.path.splitext(filename)
    out_sample = os.path.join(output_dir, f"{base}_intra_sample{ext}")

    # resume or create sample
    if os.path.exists(out_sample):
        print(f"Resuming from sampled file: {out_sample}")
        sampled = pd.read_csv(out_sample, keep_default_na=False)
    else:
        sampled = _select_pool_intra_goal(
            df,
            seed=seed,
            target_per_goal=target_per_goal,
            per_arg1_limit=per_arg1_limit,
            target_total=target_total,
        )
        if "rel" not in sampled.columns:
            sampled["rel"] = ""
        else:
            sampled["rel"] = sampled["rel"].fillna("")
        sampled.to_csv(out_sample, index=False, encoding="utf-8")
        print(f"Sample saved: {out_sample} (rows={len(sampled)})")

    print("\nAnnotation controls:")
    print("  Type S/A/R/N → Support/Attack/Rephrase/No Relationship")
    print("  Enter to skip, 'b' to go back, 'q' to save & quit.\n")

    sampled["rel"] = sampled["rel"].astype(str)
    mask_unlabeled = sampled["rel"].isna() | (sampled["rel"].str.strip().isin({"", "nan", "None"}))
    unlabeled_idx = sampled.index[mask_unlabeled].tolist()
    if unlabeled_idx:
        i = sampled.index.get_loc(unlabeled_idx[0])
    else:
        print("All sampled rows already labeled.")
        return out_sample

    labeled_since_save = 0
    while 0 <= i < len(sampled):
        row = sampled.iloc[i]
        a1 = str(row["SDGarg1"])
        a2 = str(row["SDGarg2"])
        g_left = parse_goal(row["id_arg1"])
        g_right = parse_goal(row["id_arg2"])
        print(f"\n[{i+1}/{len(sampled)}]  G={g_left}  (g1==g2? {g_left == g_right})")
        print("Arg1:", a1)
        print("Arg2:", a2)
        prev = str(row.get("rel", "")).strip()
        if prev and prev.lower() not in {"nan", "none"}:
            print(f"(already labeled: {prev})")

        key = input("Label [S/A/R/N, Enter=skip, b=back, q=quit] > ").strip().lower()

        if key == "q":
            sampled.to_csv(out_sample, index=False, encoding="utf-8")
            print(f"\nSaved & quit: {out_sample}")
            return out_sample
        if key == "b":
            i = max(0, i - 1)
            continue
        if key == "":
            i += 1
            continue

        if key in VALID_KEYS:
            sampled.iat[i, sampled.columns.get_loc("rel")] = LABEL_MAP[key]
            labeled_since_save += 1
        else:
            print("Invalid key. Use S/A/R/N, Enter, b, or q.")
            continue

        if labeled_since_save >= autosave_every:
            sampled.to_csv(out_sample, index=False, encoding="utf-8")
            print(f"(autosaved {autosave_every} labels) -> {out_sample}")
            labeled_since_save = 0

        i += 1

    sampled.to_csv(out_sample, index=False, encoding="utf-8")
    print(f"\nDone. Saved: {out_sample}")
    return out_sample

In [6]:
input_dir = r"..\\Data\\Relationships Keywords"
output_dir = r"..\\Data\\Relationships Keywords\\Annotated"
prefix = 'GLOBAL_SGD2023_'

file_names = ['intra_goalGLOBAL_SGD2023_deepseek-r1-70b.csv', 
              'intra_goalGLOBAL_SGD2023_gemma3-27b.csv']

for document in file_names:
    sample_and_annotate_intra_goal(
        input_dir,
        output_dir,
        document,
        seed=42,
        target_per_goal=25,
        per_arg1_limit=1, 
        target_total=425)

Sample saved: ..\\Data\\Relationships Keywords\\Annotated\intra_goalGLOBAL_SGD2023_deepseek-r1-70b_intra_sample.csv (rows=425)

Annotation controls:
  Type S/A/R/N → Support/Attack/Rephrase/No Relationship
  Enter to skip, 'b' to go back, 'q' to save & quit.


[1/425]  G=0  (g1==g2? True)
Arg1: The SDG policy agenda is complex. The SDGs call for lasting, long-term, directed change.
Arg2: The EU Green Deal has great potential to bring about transformation both within the EU and beyond, including the larger European and Mediterranean region, and even Africa.

[2/425]  G=0  (g1==g2? True)
Arg1: At the mid-point of the SDG agenda, we are far off target. Yet we have gained ground.
Arg2: Nation-states continue to hold the primary responsibility for achieving the SDGs.

[3/425]  G=0  (g1==g2? True)
Arg1: Virtually all governments of the world have embraced the SDGs in principle.
Arg2: The contribution of the SDGs towards a universally accepted framework for monitoring progress is critical.

[